In [ ]:
!pip install -q transformers accelerate sentencepiece wandb huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00


In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import wandb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
from huggingface_hub import login

login("<HF_TOKEN>")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================
# PROJECT PATH
# =========================
PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

BENCHMARK_PATH = f"{PROJECT_ROOT}/eval/llm_only/benchmarks/llm_only_v1.json"
PROMPT_PATH = f"{PROJECT_ROOT}/eval/llm_only/prompts/prompt_v1.txt"

RUNS_DIR = f"{PROJECT_ROOT}/eval/llm_only/runs/model_comparison"
SCORES_DIR = f"{PROJECT_ROOT}/eval/llm_only/scores"

# =========================
# W&B
# =========================
WANDB_PROJECT = "ARIA-Lite-Eval"
WANDB_GROUP = "llm_only_v1"

# =========================
# MODELS
# =========================
MODELS = {
    "qwen1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
    "mistral7b": "mistralai/Mistral-7B-Instruct-v0.2",
}

# =========================
# GENERATION SETTINGS
# =========================
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.0
DO_SAMPLE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

BENCHMARK_NAME = Path(BENCHMARK_PATH).name
PROMPT_NAME = Path(PROMPT_PATH).name

Device: cuda


In [ ]:
Path(RUNS_DIR).mkdir(parents=True, exist_ok=True)
Path(SCORES_DIR).mkdir(parents=True, exist_ok=True)

for m in MODELS:
    Path(f"{RUNS_DIR}/{m}").mkdir(parents=True, exist_ok=True)

print("Directories ready.")

Directories ready.


In [ ]:
with open(BENCHMARK_PATH, "r") as f:
    benchmark = json.load(f)

print("Loaded queries:", len(benchmark))

Loaded queries: 4


In [ ]:
with open(PROMPT_PATH, "r") as f:
    PROMPT_TEMPLATE = f.read()

print(PROMPT_TEMPLATE)

You are a biomedical research assistant.

Answer the question using only the provided context.

Question:
{query}

Context:
{sections}

Provide a scientifically accurate answer.


In [ ]:
def build_prompt(item):

    sections = "\n\n".join([
        f"[{s['section_id']}]\n{s['text']}"
        for s in item["sections"]
    ])

    return PROMPT_TEMPLATE.format(
        query=item["query"],
        sections=sections
    )

In [ ]:
def load_model(model_id):

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    # ---- Mistral 7B uses 4-bit loading for Colab safety ----
    if "mistral" in model_id.lower():

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=True
        )

    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            device_map="auto",
            trust_remote_code=True
        )

    model.eval()
    return tokenizer, model

In [ ]:
def generate_response(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[1]

    start = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.eos_token_id
    )

    latency = time.time() - start

    gen_tokens = outputs[0][input_len:]

    response = tokenizer.decode(
        gen_tokens,
        skip_special_tokens=True
    ).strip()

    output_tokens = len(tokenizer.encode(response))

    return response, latency, output_tokens

In [ ]:
wandb.login(key="<WANDB_API_KEY>")

wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    config={
        "benchmark_name": BENCHMARK_NAME,
        "benchmark_path": BENCHMARK_PATH,
        "prompt_name": PROMPT_NAME,
        "prompt_path": PROMPT_PATH,
        "temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "models": MODELS
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nishkala-aistuff (nishkala-aistuff-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
all_results = []

for model_name, model_id in MODELS.items():

    print("\n" + "="*80)
    print(f"Loading {model_name}")
    print("="*80)

    tokenizer, model = load_model(model_id)

    for item in benchmark:

        qid = item["query_id"]

        print(f"Running {qid} on {model_name}")

        prompt = build_prompt(item)

        response, latency, tokens = generate_response(
            model,
            tokenizer,
            prompt
        )

        result = {
            "timestamp": datetime.now().isoformat(),

            "benchmark_name": BENCHMARK_NAME,
            "benchmark_path": BENCHMARK_PATH,

            "prompt_name": PROMPT_NAME,
            "prompt_path": PROMPT_PATH,

            "query_id": qid,

            "model_name": model_name,
            "model_id": model_id,

            "temperature": TEMPERATURE,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": DO_SAMPLE,

            "prompt": prompt,
            "response": response,

            "latency_seconds": latency,
            "output_tokens": tokens
        }

        out_file = f"{RUNS_DIR}/{model_name}/{qid}.json"

        with open(out_file, "w") as f:
            json.dump(result, f, indent=2)

        wandb.log({
            "model_name": model_name,
            "query_id": qid,
            "latency": latency,
            "output_tokens": tokens
        })

        all_results.append(result)

    del model
    torch.cuda.empty_cache()

    print(f"Finished {model_name}")


Loading qwen1.5b


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [ ]:
df = pd.DataFrame(all_results)

csv_path = f"{SCORES_DIR}/raw_generation_stats.csv"

df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df.head()

In [ ]:
wandb.finish()